## Object Detection in MS COCO

# Full Image-Set Test Clone

This clone is stripped of outputs and has a corrected full-pipeline cell for running all partitioned images.


In [ ]:
import os
import torch
import numpy as np
from torchvision import transforms
#--------------------------------------------
from skimage.filters import gaussian
from scipy.ndimage import gaussian_filter
import cv2
import pandas as pd

import matplotlib.pyplot as plt

import shap_bpt
print('shap_bpt version:',shap_bpt.__version__)
print('shap_bpt release name:',shap_bpt.__release_name__)

## Set Config

In [ ]:
import os
from pathlib import Path
import yaml

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'shap_bpt').is_dir():
            return path
    raise FileNotFoundError('Could not find the project root from the current working directory.')

original_working_dir = Path.cwd()
project_root = find_project_root(original_working_dir)
os.chdir(project_root)
print(f'Project root: {project_root}')

config_file = "MSCOCO_mac"
# config_file = "MSCOCO_xn2"

try:
    with open(project_root / f"examples/configs/{config_file}.yaml", "r") as f:
        config = yaml.safe_load(f)
finally:
    os.chdir(original_working_dir)
    print(f'Restored working directory: {original_working_dir}')

dataset_root = config["data"]["dataset_root"]
print(f"Dataset root: {dataset_root}")

print(f"Masks Path  : {config['data']['masks_path']}")

masks_base_path = os.path.join(config['data']['masks_path'], config['data']['mask_dir'], config['data']['mask_dir_final'])

print(f"masks_base_path Path  : {masks_base_path}")

In [ ]:
import sys

scripts_dir = project_root / "examples/scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import utils_xai as utx
import utils_sam as uts
import utils_sam_2 as uts2


import importlib
importlib.reload(utx) 
importlib.reload(uts)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if ('mps' in dir(torch.backends)) and torch.backends.mps.is_available() else torch.device("cpu")
device

In [ ]:
config_compiled={}
config_compiled['device'] = device


In [ ]:
from ultralytics import YOLO
yolo_model_name = config['model']['yolo_model_name']

model = YOLO(f'{original_working_dir}/checkpoints/{yolo_model_name}.pt')
class_names = model.names

print(f'{"Num_Classes":<15}{len(class_names)}')

model.info()

from pycocotools.coco import COCO

model_preprocess = transforms.Compose(
    [transforms.ToTensor()]
)

In [ ]:
def load_image(fname,im_size=None,bg_type='noise'):
    img_ = cv2.imread(f'{fname}')
    image_to_explain = cv2.cvtColor(img_, cv2.COLOR_BGR2RGB)                #.astype(np.float32)
    if im_size is not None:
        image_to_explain         = cv2.resize(image_to_explain,im_size)     # [:,:,::-1]
    image_to_explain_preproc  = image_to_explain.copy()                     #torch.tensor(image_to_explain).to(device)# .astype(np.float32)/255.0
    np.random.seed(0)
    bkgnd0 = np.full_like(image_to_explain, 0)
    bkgnd1 = np.full_like(image_to_explain, 127)
    bkgnd2 = np.full_like(image_to_explain, 255)
    bkgnd3 = gaussian(image_to_explain, 8, channel_axis=-1)*255
    bkgnd4 = np.clip(np.random.normal(128, 128, size=image_to_explain.shape), 0, 255).astype(np.uint8)
    bkgnd4 = (gaussian(bkgnd4, 2.0, channel_axis=-1) * 255).astype(np.uint8)
    if bg_type=='black': background_image_set = np.array([bkgnd0])
    elif bg_type=='gray': background_image_set = np.array([bkgnd1])
    elif bg_type=='white': background_image_set = np.array([bkgnd2])
    elif bg_type=='blurred': background_image_set = np.array([bkgnd3])
    elif bg_type=='noise': background_image_set = np.array([bkgnd4])
    elif bg_type=='full': background_image_set = np.array([bkgnd0, bkgnd1, bkgnd2, bkgnd3, bkgnd4])
    else: raise ValueError(f'Unknown bg_type: {bg_type}')
    background_image_preproc_set = [model_preprocess(bkgnd.astype(np.float32)/255.0)
                                        for bkgnd in background_image_set]
    background_tensors = torch.cat([torch.unsqueeze(bk_p, dim=0) 
                                    for bk_p in background_image_preproc_set]).to(device)
    return image_to_explain,image_to_explain_preproc,background_image_set,background_tensors

In [ ]:
def predict_yolo(x,coco_classes_count=80,verbose=False):
    res = model.predict(source=x, verbose=verbose)[0]
    p = np.zeros(coco_classes_count)
    for cls, prob in zip(res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        cls = int(cls)
        p[cls] = max(p[cls], float(prob))
    return np.array(p)
#-----------------------------------------------------------------------
def predict_yolo_masked(masks,verbose=False):
    imglst_preds = []
    for mask in masks:
        preds = []
        for repl in background_image_set:
            # print(mask.shape, repl.shape)
            if len(mask.shape)!=3:
                mask3 = np.stack([mask,mask,mask], axis=2)
            else:
                print(mask.shape)
                mask3 = mask.copy()
            masked_image = np.where(mask3, image_to_explain, repl)
            preds.append(predict_yolo(masked_image,verbose=verbose))

        preds = np.mean(preds, axis=0)
        imglst_preds.append(preds)       
    
    return np.array(imglst_preds)

In [ ]:
def load_image_to_explain(fname,image_dir,bg_type='gray', load_gt=True):
    # global predicted_fS, predicted_f0, predicted_cls, sorted_classes, f_S, f_0,sorted_probs
    # global model_type,pretrained_model_type
    image_path = os.path.join(image_dir, f'{fname}.jpg')
    image_to_explain,image_to_explain_tensor,background_image_set,background_tensors = load_image(image_path,bg_type=bg_type)
    h,w,_ = image_to_explain.shape
    # Foreground image to be explained  
    predicted_fS = predict_yolo(image_to_explain) 
    # predicted_fS = f(torch.unsqueeze(resnet50_preprocess(image_to_explain.astype(np.float32)/255.0).to(device), dim=0))[0]
    sorted_classes = np.flip(np.argsort(predicted_fS))
    sorted_probs   = predicted_fS[sorted_classes]
    predicted_cls = sorted_classes[0]
    f_S = float(predicted_fS[predicted_cls])
    #####################
    
    predicted_f0 = [predict_yolo(bkgnd.astype(np.float32)/255.0) for bkgnd in background_image_set]
    predicted_f0 = np.mean(predicted_f0,axis=0)
    f_0          = float(predicted_f0[predicted_cls])
    # return image_to_explain,image_to_explain_tensor,background_image_set,background_tensors,predicted_fS,sorted_classes,sorted_probs,predicted_cls,f_S,predicted_f0,f_0
    return {
            "fname": fname,
            "image_to_explain": image_to_explain,
            "image_to_explain_tensor": image_to_explain_tensor,
            "background_image_set": background_image_set,
            "background_tensors": background_tensors,
            "predicted_fS": predicted_fS,
            "sorted_classes": sorted_classes,
            "sorted_probs": sorted_probs,
            "predicted_cls": predicted_cls,
            "fixed_category": class_names[predicted_cls],
            "f_S": f_S,
            "predicted_f0": predicted_f0,
            "f_0": f_0
        }
    # if load_gt:
        # image_no = int(image_path.split('\\')[-1].split('.')[0])

        # load_groundtruth(coco,image_path,fixed_category=fixed_category)
    

## Saving Results

In [ ]:
# path_results = os.path.join(original_working_dir, 'results',config['data']['mask_dir_final'])
path_results = os.path.join(config['output']['dir'], config['output']['folder'], 'xai_results')
# path_results_img = os.path.join(path_results, str(image_no))
os.makedirs(path_results, exist_ok=True)
print(f"Results will be saved in: {path_results}")

## Load Partitions

## SELECT IMAGE

In [ ]:
## Get available precomputed SAM partitions
# fetch available unique image_ids from the partitions directory
# image_ids = os.listdir(f"/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/{path_partition}")
image_ids = os.listdir(masks_base_path)
image_ids = [f.split('.')[0].split('_')[0] for f in image_ids if '_refined.npy' in f]
print(f'computed image_ids: {len(image_ids)}')

# already_computed = ['000000049091',
#  '000000000632',
#  '000000186929',
#  '000000002299',
#  '000000225757',
#  '000000171382']

# image_ids = [img_id for img_id in image_ids if img_id not in already_computed]
print(f'filtered image_ids: {len(image_ids)}')
image_ids[:10]

In [ ]:
image_dir = config["data"]["image_dir"] # Update for your image directory
annotation_file =  config["data"]["annotation_file"] # Update for your annotation file
coco = COCO(annotation_file)

categories = coco.loadCats(coco.getCatIds())
coco_categories = {cat['id']: cat['name'] for cat in categories}

In [ ]:
# image_id = '000000171382'  # Example image ID
image_id = '000000377814'  # Example image ID
# image_id = image_ids[1]  # Example image ID
image_path = os.path.join(image_dir, f'{image_id}.jpg')
print(f'Image path: {image_path}')

In [ ]:
bg_type='noise'

In [ ]:
input_data = load_image_to_explain(image_id,image_dir, bg_type=bg_type)
# fixed_category = 'tv'
fixed_category = class_names[input_data["predicted_cls"]]
# input_data['fixed_category'] = fixed_category
print('fixed_category: ',fixed_category)

In [ ]:
plt.figure(figsize=(3,3))
plt.imshow(input_data["image_to_explain"])
plt.xticks([]); plt.yticks([]); 
plt.title(f'Predicted Class: {fixed_category}'); plt.show()

In [ ]:
fig,ax = plt.subplots(1,1+len(input_data["background_image_set"]), 
                      figsize=(2*(1+len(input_data["background_image_set"])), 2))
ax[0].imshow(input_data["image_to_explain"].astype(np.uint8))
ax[0].set_title('Input')
ax[0].set_xticks([]) ; ax[0].set_yticks([])
for i,img in enumerate(input_data["background_image_set"]):
    ax[i+1].imshow(img.astype(np.uint8))
    ax[i+1].set_title(f'Replacement {i}')
    ax[i+1].set_xticks([]) ; ax[i+1].set_yticks([])
plt.show()

In [ ]:

path_results_img = os.path.join(path_results, str(input_data['fname']))
print(f"Results will be saved in: {path_results_img}")
os.makedirs(path_results_img, exist_ok=True)

In [ ]:
# image_no = 113235
def check_ground(image_id):
    image_no = int(image_id)

    image_info = coco.loadImgs(image_no)[0]
    ann_ids = coco.getAnnIds(imgIds=image_info['id'])
    annotations = coco.loadAnns(ann_ids)
    # has_segmentation = any('segmentation' in ann for ann in annotations)
    # print(f"Segmentation annotations present: {has_segmentation}")

    ann_ids = coco.getAnnIds(imgIds=image_info['id'])
    annotations = coco.loadAnns(ann_ids)
    # 
    # Check if segmentation annotations are present
    has_segmentation = any('segmentation' in ann for ann in annotations)
    print(f"Segmentation annotations present: {has_segmentation}")
    if has_segmentation:
        for ann in annotations:
            if 'segmentation' in ann:
                print(f"Segmentation Annotation: {ann['segmentation']}")
                break
            
    return has_segmentation,annotations, image_info


def get_annotation(coco,image_no,category_name=None):
    if isinstance(image_no, str):
        image_no = int(image_no.split('\\')[-1].split('.')[0])
    
    image_info = coco.loadImgs(image_no)[0]
    if category_name is None:
        annotation_ids = coco.getAnnIds(imgIds=image_info['id'])
    else:
        category_ids = coco.getCatIds(catNms=[category_name])
        annotation_ids = coco.getAnnIds(imgIds=image_info['id'], catIds=category_ids)
    annotations = coco.loadAnns(annotation_ids)
    return annotations

def create_gt(coco,input_data,category_name=None, verbose=False):
    image_no = input_data['fname']
    annotations = get_annotation(coco,image_no,category_name=category_name)
    if verbose:
        if len(annotations)>0:
            print(f"Image:{image_info['id']} has {len(annotations)} annotations")
    
    mask = np.zeros((image_info['height'], image_info['width']), dtype=np.uint8)
    category_mask = np.zeros((image_info['height'], image_info['width']), dtype=np.uint8)

    # Combine all masks for this image
    for ann in annotations:
        if 'segmentation' in ann:
            category_id = ann['category_id']  # Unique ID for object category
            # Decode the segmentation mask
            if isinstance(ann['segmentation'], list):  # Polygon format
                for seg in ann['segmentation']:
                    pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                    cv2.fillPoly(mask, [pts], color=1)  # Fill the mask polygon
                    cv2.fillPoly(category_mask, [pts], color=category_id)
            elif isinstance(ann['segmentation'], dict):  # RLE format
                rle = ann['segmentation']
                decoded_mask = coco.annToMask(ann)
                mask += decoded_mask  # Add binary mask
                category_mask[decoded_mask > 0] = category_id  # Assign category ID
    
    # Resize masks to match actual image dimensions
    if mask.shape[:2] != input_data['image_to_explain'].shape[:2]:
        # print(f"Resizing masks: Annotated={mask.shape}, Actual={image_to_explain.shape[:2]}")
        mask = cv2.resize(mask, (input_data['image_to_explain'].shape[1], input_data['image_to_explain'].shape[0]), interpolation=cv2.INTER_NEAREST)
        category_mask = cv2.resize(category_mask, (input_data['image_to_explain'].shape[1], input_data['image_to_explain'].shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask,category_mask,annotations

In [ ]:
def load_groundtruth(coco,data,fixed_category=None):
    # global ground_truth,weighted_ground_truth,annotations
    mask,ground_truth,annotations = create_gt(coco,data,category_name = fixed_category)
    weighted_ground_truth = gaussian_filter(ground_truth.astype(float), 16) * ground_truth
    ground_truth.dtype = 'bool'
    return { "mask": mask, "ground_truth":
            ground_truth, "weighted_ground_truth":
            weighted_ground_truth,
            "annotations": annotations }


In [ ]:
has_segmentation,annotations,image_info = check_ground(image_id)

In [ ]:
# fixed_category = None
fixed_category = input_data['fixed_category']


gt = load_groundtruth(coco,input_data,fixed_category=fixed_category)

plt.imshow(gt['ground_truth'], cmap='gray')
plt.xticks([]); plt.yticks([]); plt.title(f'Ground Truth Mask for {fixed_category}'); plt.show()

In [ ]:
# # Perform inference on an image
# def plot_predictions(image, results, category_name=None, filter_preds=True,
#                      line_thickness=2, exp_type='demo', save_fig=False, fig_size=(3, 3),
#                      title=None, selected_ext='png', destroy_fig=False):
#     image_ = image.copy()
#     plt.figure(figsize=fig_size)
#     plt.imshow(image_); plt.axis('off')
#     for result in results:
#         # Access detected classes, confidences, and boxes
#         class_ids = result.boxes.cls.cpu().numpy()  # Class IDs
#         scores = result.boxes.conf.cpu().numpy()   # Confidence scores
#         boxes = result.boxes.xyxy.cpu().numpy()    # Bounding boxes in xyxy format
#         labels = model.names                       # Class labels (MS COCO classes)
#         # Draw bounding boxes and labels on the image
#         for box, class_id, score in zip(boxes, class_ids, scores):
#             label = labels[int(class_id)]
#             confidence = f"{score:.2f}"
#             x1, y1, x2, y2 = map(int, box)  # Bounding box coordinates
#             if filter_preds and category_name and label != category_name:
#                 continue
#             width,height = x2 - x1, y2 - y1
#             plt.gca().add_patch(plt.Rectangle((x1, y1), width, height, edgecolor='darkred', facecolor='none', linewidth=line_thickness))
#             plt.text(x1, y1 - 5, f"{label} {confidence}", color='white', fontsize=12, bbox=dict(facecolor='darkred', alpha=0.5))
#     plt.show()


In [ ]:
importlib.reload(utx)
results = model.predict(input_data['image_to_explain'],verbose=True)

In [ ]:
top_classes_n = 5

top_k_classes = utx.get_top_k_classes(results,model.names, k=5)
print("Top-k Classes (Class ID, Confidence):")
print(top_k_classes)

# print([class_names[int(cls)] for cls, _,_ in top_k_classes])
# print('-'*120)
# print('top_k_classes \t', top_k_classes)

In [ ]:
fixed_category

In [ ]:
utx.plot_predictions(input_data['image_to_explain'],model,results,
                     category_name = fixed_category,
                     fig_size=(5,5),
                     save_fig=False)

In [ ]:
# MASKING FUNCTION
def predict_yolo_masked(masks):
    imglst_preds = []
    for mask in masks:
        preds = []
        for repl in input_data['background_image_set']:
            if len(mask.shape)==2:
                mask3 = np.stack([mask,mask,mask], axis=2)
            else:
                mask3 = mask.copy()
            masked_image = np.where(mask3, input_data['image_to_explain'], repl)
            preds.append(predict_yolo(masked_image))
        preds = np.mean(preds, axis=0)
        imglst_preds.append(preds)       
    return np.array(imglst_preds)

# def predict_yolo(x, coco_classes_count=80,verbose=False):
#     res = model.predict(source=x, verbose=verbose)[0]
#     p = np.zeros(coco_classes_count)

#     for cls, prob in zip(res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
#         cls = int(cls)
#         # p[cls] = max(p[cls], float(prob)) # 
#         p[cls] = sum([p[cls], float(prob)]) # with sum of each class probability

#     return p

def predict_yolo(model: YOLO, x, coco_classes_count: int = 80, verbose: bool = False, aggregate: str = "sum"):
    result = model.predict(source=x, verbose=verbose)[0]
    p = np.zeros(coco_classes_count)
    for cls, prob in zip(result.boxes.cls.cpu().numpy(), result.boxes.conf.cpu().numpy()):
        cls = int(cls)
        if aggregate == "max":
            p[cls] = max(p[cls], float(prob))
        else:
            p[cls] += float(prob)
    return p

In [ ]:
importlib.reload(utx)
results = model.predict(input_data['image_to_explain'], verbose=True)
top_k_classes = utx.get_top_k_classes(results, model.names, k=5)

predicted_fS = utx.predict_yolo(model,input_data['image_to_explain'])
predicted_cls = int(np.argmax(predicted_fS))

print("YOLO top box:", top_k_classes[0])
print("predict_yolo top class:", predicted_cls, model.names[predicted_cls], predicted_fS[predicted_cls])

In [ ]:
print(input_data['image_to_explain'].shape)
bg_ls = np.zeros((100,input_data['image_to_explain'].shape[0],input_data['image_to_explain'].shape[1],input_data['image_to_explain'].shape[2]))
# bg_ls = np.random.randint(0, 255, (50,426, 640, 3), dtype=np.uint8)
print(bg_ls.shape)
pred = predict_yolo_masked(bg_ls)
print(pred.shape)
torch.cuda.empty_cache()

In [ ]:
num_explained_classes        = 1
MAX_EVALS_BUDGET             = 500
explainer   = shap_bpt.Explainer(predict_yolo_masked, input_data['image_to_explain'], num_explained_classes=num_explained_classes, verbose=True)

In [ ]:
xai_classes = [
    (int(cls_id), class_names[int(cls_id)], float(score))
    for cls_id, score in zip(explainer.output_indexes, explainer.base_nuN)
]
explained_class_id = int(explainer.output_indexes[0])
explained_class = class_names[explained_class_id]

input_data['explained_class'] = explained_class

print('XAI explained classes (Class ID, Name, Confidence):', xai_classes)
if explained_class != fixed_category:
    print(f'WARNING: fixed_category={fixed_category!r} but XAI top explained class={explained_class!r}')

In [ ]:
importlib.reload(utx)
detection_summary = utx.save_yolo_predictions(results,input_data,config,has_segmentation,top_k_classes)
detection_summary

In [ ]:
shap_values = {}

In [ ]:
shap_values['BPT'] = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT',batch_size=16)
# shap_values['AA'] = explainer.explain_instance(MAX_EVALS_BUDGET, method='AA',  batch_size=16)

In [ ]:
shap_bpt.plot_owen_values(explainer, [shap_values['BPT']],class_names, names=['BPT'])

In [ ]:
## for shapbpt 1.0
# print('Expected Shapley explanation: ', explainer.base_f_S[0] - explainer.base_f_0[0])
# print('Computed Shapley explanation: ', np.sum(shap_values['BPT'][0]))

In [ ]:
## for shapbpt 1.1
print('Expected Shapley explanation: ', explainer.base_nuN[0] - explainer.base_nu0[0])
print('Computed Shapley explanation: ', np.sum(shap_values['BPT'][0]))

## Genearte/LOAD SAM

In [ ]:
verbose = True

## LOAD Partitions

In [ ]:
def build_bpt_from_image(img_, partitions, verbose=True):
    bptrees = {}
    bptrees['BPT'] = shap_bpt.build_bpt_from_image(img_)

    for versions in partitions.keys():
        bptrees[versions] = shap_bpt.build_bpt_from_image(img_, prebuilt_partitions=partitions[versions])
        if verbose:
            print(f"Built BPT for {versions} partition: {bptrees[versions]}")
    return bptrees

## SAM Guided BPT

In [ ]:
MAX_EVALS_BUDGET = 100

xai_methods_list = ['BPT','sam','coverage','compact', 'filled', 'refined']


In [ ]:
importlib.reload(uts2)
partitions,partition_capped = {},{}

for version in ['sam','coverage','compact', 'filled', 'refined']:
    partitions[version], partition_capped[version] = uts2.load_refine_partitions(input_data['image_to_explain'], masks_base_path, image_id, partition_type=version, verbose=verbose)

In [ ]:
importlib.reload(uts)
uts.plot_masks([partitions['sam'], partitions['coverage'], partitions['compact'], partitions['filled'], partitions['refined']], 
               mask_types=['sam', 'coverage', 'compact', 'filled', 'refined'],version='mask', verbose=False)
uts.plot_masks([partition_capped['sam'], partition_capped['coverage'], partition_capped['compact'], partition_capped['filled'], partition_capped['refined']], 
               mask_types=['sam_cap', 'coverage_cap', 'compact_cap', 'filled_cap', 'refined_cap'],version='mask_capped', verbose=False)

In [ ]:
import pandas as pd
partition_results = uts2.load_json_partitions(masks_base_path, image_id, verbose=False)

print('imgs', len(image_ids))

df_partitions = []
for imgs in image_ids:
    df_partitions_data = {}
    partition_results = uts2.load_json_partitions(masks_base_path, imgs, verbose=False)
    df_partitions_data = {
        'image_id': partition_results['name'].split('.')[0],
        'n_sam_masks': partition_results.get('n_sam_masks', 0),
        'n_coverage_masks': partition_results.get('n_coverage_masks', 0),
        'n_refined_instances': partition_results.get('n_refined_instances', 0),
        'n_unique_filler': partition_results.get('n_unique_filler', 0),
        'total_time_sec': partition_results.get('total_time_sec', 0.0),
        'time_sam': partition_results.get('time_sam', 0.0),
        'time_coverage': partition_results.get('time_coverage', 0.0),
        'time_compact': partition_results.get('time_compact', 0.0),
        'time_filler': partition_results.get('time_filler', 0.0),
        'time_refined': partition_results.get('time_refined', 0.0),
        'steps': partition_results.get('steps', 0),
        'non_zero_px': partition_results.get('non_zero_px', 0),
        'total_px': partition_results.get('total_px', 0),
        'coverage': partition_results.get('coverage', 0.0),
        
    }
    df_partitions.append(df_partitions_data)
df_partitions_final = pd.DataFrame(df_partitions)
path_partition_summary = os.path.join(path_results, 'partition_summary.csv')
df_partitions_final.to_csv(path_partition_summary , index=False)
print('Partition Path Summary saved to: \n', path_partition_summary)

In [ ]:
df_partitions_final

In [ ]:
partitions, partition_capped = {}, {}
for version in xai_methods_list:
    if version == 'BPT':
        pass
    else:
        print('version: ', version  )
        partitions[version], partition_capped[version] = uts2.load_refine_partitions(input_data['image_to_explain'], 
                                                                                     masks_base_path, image_id,
                                                                                     partition_type=version,
                                                                                     verbose=verbose)
        bptree = shap_bpt.build_bpt_from_image(input_data['image_to_explain'],
                                            prebuilt_partitions=partition_capped[version])
    
    shap_values[version] = explainer.explain_instance(MAX_EVALS_BUDGET,bpt =None if version == 'BPT' else bptree, method='BPT',batch_size=16)
    shap_bpt.plot_owen_values(explainer, shap_values[version], class_names, names=[version])

In [ ]:
shap_values.keys()

In [ ]:
shap_bpt.plot_owen_values(explainer, 
                          [shap_values['BPT'],
                           shap_values['sam'],
                           shap_values['coverage'],
                           shap_values['compact'],
                           shap_values['filled'],
                           shap_values['refined']],
          class_names, names=['BPT','SAM_S','SAM_Cov','SAM_Com','SAM_Fil','SAM_Ref'])

In [ ]:
MAX_EVALS_BUDGET = 10

In [ ]:
importlib.reload(utx)
utx.plot_xai(input_data,partitions,shap_values, model, results, image_id)

## Evaluation

## RUN FULL PIPELINE

In [ ]:
# Full image-set run switch for this cloned test notebook.
run_fullpipeline = True


In [ ]:
# Full image-set test run. This cell mirrors the current single-image method set:
# BPT, SAM_S, SAM_R, BPT_SAM_S, and BPT_SAM_R. No flag grid is swept.
# from time import time
import time


save_plots = True
compute_auc = True
destroy_figs = True
verbose = True
verbose_level = 'low' # low, medium, high

skip_existing_reports = False

MAX_EVALS_BUDGET = 500
full_image_limit = None  # set to an int for a quick smoke test, e.g. 2

failures = []
completed_reports = []
auc_table_rows = []
# path_results = os.path.join(original_working_dir, 'results')
path_results = os.path.join(config['output']['dir'], config['output']['folder'], 'xai_results')

os.makedirs(path_results, exist_ok=True)

importlib.reload(uts)
importlib.reload(uts2)
importlib.reload(utx)


if run_fullpipeline:
    import json
    import traceback
    from tqdm.auto import tqdm

    num_explained_classes = 4
    eval_batch_size = 16
    auc_batch_size = 4

    run_image_ids = list(image_ids)
    if full_image_limit is not None:
        run_image_ids = run_image_ids[:full_image_limit]

    for image_id_raw in tqdm(run_image_ids, desc='Full image-set XAI'):
        image_id = f"{int(image_id_raw):012d}"
        image_no = int(image_id)
        report_html = os.path.join(path_results, f'shapbpt_report_{image_no}.html')

        if skip_existing_reports and os.path.exists(report_html):
            if verbose:
                print(f'Skipping existing report: {report_html}')
            completed_reports.append(report_html)
            continue

        if verbose:
            print('=' * 100)
        
        image_dir = config['data']['image_dir']
        image_path = os.path.join(image_dir, f'{image_id}.jpg')

        input_data = load_image_to_explain(image_id,image_dir, bg_type=bg_type)

        # fixed_category = class_names[predicted_cls]
        fixed_category = class_names[input_data["predicted_cls"]]
        if verbose:
            print(f'Image: {image_id} - Fixed Category: {fixed_category}')
        
        image_info = coco.loadImgs(image_no)[0]
        ann_ids = coco.getAnnIds(imgIds=image_info['id'])
        annotations = coco.loadAnns(ann_ids)
        # has_segmentation = any('segmentation' in ann for ann in annotations)

        results = model.predict(input_data['image_to_explain'], verbose=False)
        yolo_speed = getattr(results[0], 'speed', {}) if results else {}
        top_k_classes = utx.get_top_k_classes(results, model.names, k=top_classes_n)

        path_results_img = os.path.join(path_results, str(image_no))
        os.makedirs(path_results_img, exist_ok=True)
        sam_path = os.path.join(masks_base_path, f'{image_id}_sam.npy')
        sorted_path = os.path.join(masks_base_path,  f'{image_id}_sorted.npy')
        refined_path = os.path.join(masks_base_path, f'{image_id}_refined.npy')

        #-------------  XAI PART  ----------------
        explainer   = shap_bpt.Explainer(predict_yolo_masked, input_data['image_to_explain'],
                                            num_explained_classes=num_explained_classes,
                                            verbose=verbose and verbose_level=='high')
        predicted_cls = explainer.output_indexes[0]
        f_S = explainer.base_nuN[0]
        f_0 = explainer.base_nu0[0]
        
        xai_classes = [
            (int(cls_id), class_names[int(cls_id)], float(score))
            for cls_id, score in zip(explainer.output_indexes, explainer.base_nuN) #base_f_S
        ]
        explained_class_id = int(explainer.output_indexes[0])
        explained_class = class_names[explained_class_id]
        if verbose and verbose_level=='high':
            print('XAI explained classes (Class ID, Name, Confidence):', xai_classes)
            if explained_class != fixed_category:
                print(f'WARNING: fixed_category={fixed_category!r} but XAI top explained class={explained_class!r}')

        input_data['explained_class'] = explained_class

        path_results_img = os.path.join(path_results, str(image_no))
        os.makedirs(path_results_img, exist_ok=True)
        ## SAVE YOLO PREDICTIONS
        detection_summary = utx.save_yolo_predictions(results,input_data,config,has_segmentation,top_k_classes)

        partitions, partition_capped = {}, {}
        shap_values = {}
        auc_records = []
        time_exp = {}
        for version in xai_methods_list:
            if version == 'BPT':
                bptree= None
            else:
                partitions[version], partition_capped[version] = uts2.load_refine_partitions(input_data['image_to_explain'],
                                                                                              masks_base_path, image_id, partition_type=version)
                bptree = shap_bpt.build_bpt_from_image(input_data['image_to_explain'],
                                                    prebuilt_partitions=partition_capped[version])
            time_start = time.time()

            shap_values[version] = explainer.explain_instance(MAX_EVALS_BUDGET,
                                                                bpt =bptree,
                                                                method='BPT',
                                                                batch_size=16)
            time_exp[version] = time.time() - time_start
            if verbose and verbose_level=='high':
                print(f'Computed Shapley explanation: {version} - {np.sum(shap_values[version][0])}')

            #  LOAD PARTITION RESULTS
            partition_results = uts2.load_json_partitions(masks_base_path, image_id, verbose=verbose and verbose_level=='high')

            

            figure_name = f'{version}_{image_no}.png'
            utx.plot_single_attributions(
                shap_values[version],
                path_results_img,
                figure_name,
                robust_percentile=99.9985,
                save_plot=save_plots,
                destroy_fig=destroy_figs,
            )

            auc_records.append({
                'label': version,
                'shap_values': shap_values[version][0],
                'time_partition': partition_results.get(f'time_{version}', None),
                'time_exp': time_exp[version],
                
            })
        utx.plot_xai(input_data,partitions,shap_values, model, results, image_id, destroy_fig=True,
                      save_path=path_results_img, save_fig=True)
        if compute_auc:
            auc_results = []
            auc_results = utx.compute_auc_results(auc_records,predict_yolo_masked,
                                                f_S,f_0,predicted_cls,
                                                batch_size=auc_batch_size,
                                                verbose=verbose and verbose_level=='high',
                                                )
            utx.plot_auc_results(auc_results,path_results_img,save_plot=save_plots,
                                image_no=image_no,
                                destroy_figs=destroy_figs,
                                )
            data = utx.auc_results_to_rows(
                auc_results,
                image_no=image_no,
                image_id=image_id,
                fixed_category=fixed_category,
                f_S=f_S,
                f_0=f_0,
            )
            auc_table_rows.extend(data)

    if completed_reports:
        all_report_path, all_report_df = utx.build_all_images_html_report(
            path_results,
            os.path.join(path_results, 'shapbpt_all_images_report.html'),
        )
        print(f'All-images report: {all_report_path}')

    if auc_table_rows:
        auc_all_df = pd.DataFrame(auc_table_rows)
        auc_all_path = os.path.join(path_results, 'auc_results_all_images_current_run.csv')
        auc_all_df.to_csv(auc_all_path, index=False)
        print(f'Current-run AUC table: {auc_all_path}')

    if failures:
        failures_path = os.path.join(path_results, 'full_image_set_failures.json')
        with open(failures_path, 'w', encoding='utf-8') as f:
            json.dump(failures, f, indent=2)
        print(f'Failures saved to: {failures_path}')

print(f'Completed reports: {len(completed_reports)}')
print(f'Failures: {len(failures)}')


In [ ]:
if auc_table_rows:
        auc_all_df = pd.DataFrame(auc_table_rows)
        auc_all_path = os.path.join(path_results, 'auc_results_all_images_current_run.csv')
        auc_all_df.to_csv(auc_all_path, index=False)
        print(f'Current-run AUC table: {auc_all_path}')

In [ ]:
auc_all_df

## END

In [ ]:
# path_results, 'auc_results_all_images_current_run_', config["data"]["mask_dir_final"], '.csv'

In [ ]:
## Aggregate AUC table and box plots across all computed images
from pathlib import Path

# path_results = Path(original_working_dir) / 'results'
no_images_computed = len(auc_all_df['image_no'].unique())

current_run_path = os.path.join(path_results, 'auc_results_all_images_current_run_',
                                 config["data"]["mask_dir_final"], f'{MAX_EVALS_BUDGET}_{no_images_computed}.csv')

auc_all_path = os.path.join(path_results, f'auc_results_all_images_{MAX_EVALS_BUDGET}_{no_images_computed}.csv')
auc_summary_path = os.path.join(path_results, f'auc_results_summary_{MAX_EVALS_BUDGET}_{no_images_computed}.csv')
box_plot_path = os.path.join(path_results, f'auc_results_boxplots_{MAX_EVALS_BUDGET}_{no_images_computed}.png')


if 'auc_table_rows' in globals() and len(auc_table_rows) > 0:
    auc_all = pd.DataFrame(auc_table_rows)
elif current_run_path.exists():
    auc_all = pd.read_csv(current_run_path)
else:
    raise FileNotFoundError(f'No combined AUC table found at {current_run_path}. Run the full-pipeline cell first.')

auc_all['method'] = pd.Categorical(
    auc_all['method'],
    categories=['BPT', 'sam', 'coverage', 'compact', 'filled','refined'],
    ordered=True,
)
auc_all = auc_all.sort_values(['method', 'image_no']).reset_index(drop=True)

summary = (
    auc_all
    .groupby('method', observed=True)
    .agg(
        images=('image_no', 'nunique'),
        auc_ins_mean=('auc_ins', 'mean'),
        auc_ins_std=('auc_ins', 'std'),
        auc_ins_median=('auc_ins', 'median'),
        auc_del_mean=('auc_del', 'mean'),
        auc_del_std=('auc_del', 'std'),
        auc_del_median=('auc_del', 'median'),
    )
    .reset_index()
)

display(auc_all)
display(summary)

auc_all.to_csv(auc_all_path, index=False)
summary.to_csv(auc_summary_path, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
plot_data = [
    (axes[0], 'auc_ins', '$\mathit{AUC}^{+}$ across images', 'Higher is better'),
    (axes[1], 'auc_del', '$\mathit{AUC}^{-}$ across images', 'Lower is better'),
]

methods = [m for m in ['BPT', 'sam', 'coverage', 'compact', 'filled','refined'] if m in set(auc_all['method'].dropna().astype(str))]
for ax, metric, title, ylabel in plot_data:
    values = [auc_all.loc[auc_all['method'].astype(str) == method, metric].dropna().values for method in methods]
    ax.boxplot(values, labels=methods, showmeans=True, patch_artist=True)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=25)

plt.suptitle(f'Aggregate AUC Results Across {len(auc_all.image_no.unique())} Images & Budget: {MAX_EVALS_BUDGET}', fontsize=16)
plt.tight_layout()

plt.savefig(box_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved aggregate AUC table: {auc_all_path}')
print(f'Saved aggregate summary: {auc_summary_path}')
print(f'Saved box plot: {box_plot_path}')

In [ ]:
auc_all_df.head()

## Time comparison for Partition only


In [ ]:
## Aggregate AUC table and box plots across all computed images
from pathlib import Path

no_images_computed = len(auc_all_df['image_no'].unique())

current_run_path = Path(path_results) / 'auc_results_all_images_current_run_' / config["data"]["mask_dir_final"] / f'{MAX_EVALS_BUDGET}_{no_images_computed}.csv'
auc_all_path = Path(path_results) / f'auc_results_all_images_{MAX_EVALS_BUDGET}_{no_images_computed}.csv'
auc_summary_path = Path(path_results) / f'auc_results_summary_{MAX_EVALS_BUDGET}_{no_images_computed}.csv'
box_plot_path = Path(path_results) / f'auc_results_boxplots_{MAX_EVALS_BUDGET}_{no_images_computed}.png'
partition_summary_path = Path(path_results) / 'partition_summary.csv'
partition_time_summary_path = Path(path_results) / f'partition_time_summary_{no_images_computed}.csv'

if 'auc_table_rows' in globals() and len(auc_table_rows) > 0:
    auc_all = pd.DataFrame(auc_table_rows)
elif current_run_path.exists():
    auc_all = pd.read_csv(current_run_path)
else:
    raise FileNotFoundError(f'No combined AUC table found at {current_run_path}. Run the full-pipeline cell first.')

if 'df_partitions_final' in globals() and len(df_partitions_final) > 0:
    partition_summary_df = df_partitions_final.copy()
elif partition_summary_path.exists():
    partition_summary_df = pd.read_csv(partition_summary_path)
else:
    raise FileNotFoundError(f'No partition summary found at {partition_summary_path}. Run the partition-summary cell first.')

auc_all['method'] = pd.Categorical(
    auc_all['method'],
    categories=['BPT', 'sam', 'coverage', 'compact', 'filled', 'refined'],
    ordered=True,
)
auc_all = auc_all.sort_values(['method', 'image_no']).reset_index(drop=True)

summary = (
    auc_all
    .groupby('method', observed=True)
    .agg(
        images=('image_no', 'nunique'),
        auc_ins_mean=('auc_ins', 'mean'),
        auc_ins_std=('auc_ins', 'std'),
        auc_ins_median=('auc_ins', 'median'),
        auc_del_mean=('auc_del', 'mean'),
        auc_del_std=('auc_del', 'std'),
        auc_del_median=('auc_del', 'median'),
    )
    .reset_index()
)

partition_time_cols = [
    ('time_sam', 'SAM'),
    ('time_coverage', 'Coverage'),
    ('time_compact', 'Compact'),
    ('time_filler', 'Filler'),
    ('time_refined', 'Refined'),
]
partition_time_cols = [(col, label) for col, label in partition_time_cols if col in partition_summary_df.columns]
if not partition_time_cols:
    raise ValueError(f'No partition time columns found in {partition_summary_path}')

partition_time_summary = (
    partition_summary_df[[col for col, _ in partition_time_cols]]
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
    .T
    .reset_index()
    .rename(columns={'index': 'time_component'})
)
partition_time_summary['time_component'] = partition_time_summary['time_component'].map(dict(partition_time_cols))

display(auc_all)
display(summary)
display(partition_time_summary)

auc_all.to_csv(auc_all_path, index=False)
summary.to_csv(auc_summary_path, index=False)
partition_time_summary.to_csv(partition_time_summary_path, index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
auc_plot_data = [
    (axes[0], 'auc_ins', '$\mathit{AUC}^{+}$ across images', 'Higher is better'),
    (axes[1], 'auc_del', '$\mathit{AUC}^{-}$ across images', 'Lower is better'),
]

methods = [m for m in ['BPT', 'sam', 'coverage', 'compact', 'filled', 'refined'] if m in set(auc_all['method'].dropna().astype(str))]
for ax, metric, title, ylabel in auc_plot_data:
    values = [auc_all.loc[auc_all['method'].astype(str) == method, metric].dropna().values for method in methods]
    ax.boxplot(values, labels=methods, showmeans=True, patch_artist=True)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=25)

partition_time_values = [
    partition_summary_df[col].dropna().astype(float).values
    for col, _ in partition_time_cols
]
partition_time_labels = [label for _, label in partition_time_cols]
axes[2].boxplot(partition_time_values, labels=partition_time_labels, showmeans=True, patch_artist=True)
axes[2].set_title('Time comparison for Partition only')
axes[2].set_ylabel('Seconds (lower is better)')
axes[2].grid(axis='y', alpha=0.25)
axes[2].tick_params(axis='x', rotation=25)

plt.suptitle(f'Aggregate AUC and Partition Time Across {len(auc_all.image_no.unique())} Images & Budget: {MAX_EVALS_BUDGET}', fontsize=16)
plt.tight_layout()

plt.savefig(box_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved aggregate AUC table: {auc_all_path}')
print(f'Saved aggregate summary: {auc_summary_path}')
print(f'Saved partition time summary: {partition_time_summary_path}')
print(f'Saved box plot: {box_plot_path}')

## Time comparison Plot with Partition and Expplanation


In [ ]:
## Time comparison plot with partition and explanation/evaluation time
from pathlib import Path

method_order = ['BPT', 'sam', 'coverage', 'compact', 'filled', 'refined']
no_images_computed = len(auc_all_df['image_no'].unique())

partition_summary_path = Path(path_results) / 'partition_summary.csv'
time_plot_path = Path(path_results) / f'time_comparison_partition_explanation_{MAX_EVALS_BUDGET}_{no_images_computed}.png'
time_summary_path = Path(path_results) / f'time_comparison_summary_{MAX_EVALS_BUDGET}_{no_images_computed}.csv'

if 'df_partitions_final' in globals() and len(df_partitions_final) > 0:
    partition_summary_df = df_partitions_final.copy()
elif partition_summary_path.exists():
    partition_summary_df = pd.read_csv(partition_summary_path)
else:
    raise FileNotFoundError(f'No partition summary found at {partition_summary_path}. Run the partition-summary cell first.')

if 'auc_all_df' in globals() and len(auc_all_df) > 0:
    time_auc_df = auc_all_df.copy()
elif 'auc_all' in globals() and len(auc_all) > 0:
    time_auc_df = auc_all.copy()
else:
    raise ValueError('auc_all_df or auc_all is required to plot time_exp and time_eval.')

for required_col in ['method', 'time_exp', 'time_eval']:
    if required_col not in time_auc_df.columns:
        raise ValueError(f'Missing required column in AUC dataframe: {required_col}')

time_auc_df['method'] = pd.Categorical(time_auc_df['method'], categories=method_order, ordered=True)
time_auc_df = time_auc_df.sort_values(['method', 'image_no']).reset_index(drop=True)

partition_time_cols = [
    ('time_sam', 'SAM'),
    ('time_coverage', 'Coverage'),
    ('time_compact', 'Compact'),
    ('time_filler', 'Filler'),
    ('time_refined', 'Refined'),
]
partition_time_cols = [(col, label) for col, label in partition_time_cols if col in partition_summary_df.columns]
if not partition_time_cols:
    raise ValueError(f'No partition time columns found in {partition_summary_path}')

methods = [method for method in method_order if method in set(time_auc_df['method'].dropna().astype(str))]

partition_values = [partition_summary_df[col].dropna().astype(float).values for col, _ in partition_time_cols]
partition_labels = [label for _, label in partition_time_cols]
exp_values = [time_auc_df.loc[time_auc_df['method'].astype(str) == method, 'time_exp'].dropna().astype(float).values for method in methods]
eval_values = [time_auc_df.loc[time_auc_df['method'].astype(str) == method, 'time_eval'].dropna().astype(float).values for method in methods]

partition_time_summary = (
    partition_summary_df[[col for col, _ in partition_time_cols]]
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
    .T
    .reset_index()
    .rename(columns={'index': 'time_name'})
)
partition_time_summary['group'] = 'partition'
partition_time_summary['time_name'] = partition_time_summary['time_name'].map(dict(partition_time_cols))

explanation_time_summary = (
    time_auc_df
    .groupby('method', observed=True)
    .agg(
        images=('image_no', 'nunique'),
        time_exp_mean=('time_exp', 'mean'),
        time_exp_std=('time_exp', 'std'),
        time_exp_median=('time_exp', 'median'),
        time_eval_mean=('time_eval', 'mean'),
        time_eval_std=('time_eval', 'std'),
        time_eval_median=('time_eval', 'median'),
    )
    .reset_index()
)

time_summary = {
    'partition_time_summary': partition_time_summary,
    'explanation_time_summary': explanation_time_summary,
}

display(partition_time_summary)
display(explanation_time_summary)

# Save as two compact tables in one CSV-friendly long format.
partition_time_summary.to_csv(time_summary_path.with_name(time_summary_path.stem + '_partition.csv'), index=False)
explanation_time_summary.to_csv(time_summary_path.with_name(time_summary_path.stem + '_explanation.csv'), index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4), sharey=False)

axes[0].boxplot(partition_values, labels=partition_labels, showmeans=True, patch_artist=True)
axes[0].set_title('Partition time only')
axes[0].set_ylabel('Seconds (lower is better)')
axes[0].grid(axis='y', alpha=0.25)
axes[0].tick_params(axis='x', rotation=25)

axes[1].boxplot(exp_values, labels=methods, showmeans=True, patch_artist=True)
axes[1].set_title('Explanation compute time')
axes[1].set_ylabel('Seconds (lower is better)')
axes[1].grid(axis='y', alpha=0.25)
axes[1].tick_params(axis='x', rotation=25)

axes[2].boxplot(eval_values, labels=methods, showmeans=True, patch_artist=True)
axes[2].set_title('AUC evaluation time')
axes[2].set_ylabel('Seconds (lower is better)')
axes[2].grid(axis='y', alpha=0.25)
axes[2].tick_params(axis='x', rotation=25)

plt.suptitle(f'Time Comparison Across {no_images_computed} Images & Budget: {MAX_EVALS_BUDGET}', fontsize=16)
plt.tight_layout()
plt.savefig(time_plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved time comparison plot: {time_plot_path}')
print(f'Saved partition time summary: {time_summary_path.with_name(time_summary_path.stem + "_partition.csv")}')
print(f'Saved explanation time summary: {time_summary_path.with_name(time_summary_path.stem + "_explanation.csv")}')

## 

## Sum up total time
- this includes the partition time and explanation time

In [ ]:
## Sum up total time: partition generation + explanation compute time
from pathlib import Path

method_order = ['BPT', 'sam', 'coverage', 'compact', 'filled', 'refined']
partition_time_by_method = {
    'BPT': None,          # No external saved partition-generation step for plain BPT.
    'sam': 'time_sam',
    'coverage': 'time_coverage',
    'compact': 'time_compact',
    'filled': 'time_filler',
    'refined': 'time_refined',
}

no_images_computed = len(auc_all_df['image_no'].unique())
partition_summary_path = Path(path_results) / 'partition_summary.csv'
total_time_table_path = Path(path_results) / f'total_time_by_method_{MAX_EVALS_BUDGET}_{no_images_computed}.csv'
total_time_plot_path = Path(path_results) / f'total_time_by_method_{MAX_EVALS_BUDGET}_{no_images_computed}.png'

if 'df_partitions_final' in globals() and len(df_partitions_final) > 0:
    partition_summary_df = df_partitions_final.copy()
elif partition_summary_path.exists():
    partition_summary_df = pd.read_csv(partition_summary_path)
else:
    raise FileNotFoundError(f'No partition summary found at {partition_summary_path}. Run the partition-summary cell first.')

if 'auc_all_df' in globals() and len(auc_all_df) > 0:
    total_time_auc_df = auc_all_df.copy()
elif 'auc_all' in globals() and len(auc_all) > 0:
    total_time_auc_df = auc_all.copy()
else:
    raise ValueError('auc_all_df or auc_all is required to compute total method time.')

for required_col in ['method', 'time_exp']:
    if required_col not in total_time_auc_df.columns:
        raise ValueError(f'Missing required column in AUC dataframe: {required_col}')

def normalize_image_id(value):
    text = str(value)
    if text.endswith('.0'):
        text = text[:-2]
    return text.zfill(12)

auc_image_col = 'image_id' if 'image_id' in total_time_auc_df.columns else 'image_no'
partition_image_col = 'image_id' if 'image_id' in partition_summary_df.columns else None
if partition_image_col is None:
    raise ValueError('partition_summary_df must contain image_id to align partition and explanation time.')

total_time_auc_df['image_id_str'] = total_time_auc_df[auc_image_col].apply(normalize_image_id)
partition_summary_df['image_id_str'] = partition_summary_df[partition_image_col].apply(normalize_image_id)
total_time_auc_df['method'] = pd.Categorical(total_time_auc_df['method'], categories=method_order, ordered=True)

time_rows = []
for method in method_order:
    method_df = total_time_auc_df[total_time_auc_df['method'].astype(str) == method].copy()
    if method_df.empty:
        continue

    partition_col = partition_time_by_method.get(method)
    method_df['partition_time_sec'] = 0.0
    if partition_col is not None:
        if partition_col not in partition_summary_df.columns:
            raise ValueError(f'Missing partition time column: {partition_col}')
        method_df = method_df.merge(
            partition_summary_df[['image_id_str', partition_col]],
            on='image_id_str',
            how='left',
        )
        method_df['partition_time_sec'] = method_df[partition_col].fillna(0).astype(float)

    method_df['explanation_time_sec'] = method_df['time_exp'].fillna(0).astype(float)
    method_df['total_time_sec'] = method_df['partition_time_sec'] + method_df['explanation_time_sec']

    time_rows.append({
        'method': method,
        'images': method_df['image_id_str'].nunique(),
        'partition_time_mean': method_df['partition_time_sec'].mean(),
        'partition_time_sum': method_df['partition_time_sec'].sum(),
        'explanation_time_mean': method_df['explanation_time_sec'].mean(),
        'explanation_time_sum': method_df['explanation_time_sec'].sum(),
        'total_time_mean': method_df['total_time_sec'].mean(),
        'total_time_sum': method_df['total_time_sec'].sum(),
        'total_time_median': method_df['total_time_sec'].median(),
    })

total_time_by_method = pd.DataFrame(time_rows)
total_time_by_method['method'] = pd.Categorical(total_time_by_method['method'], categories=method_order, ordered=True)
total_time_by_method = total_time_by_method.sort_values('method').reset_index(drop=True)

display(total_time_by_method)
total_time_by_method.to_csv(total_time_table_path, index=False)

x = np.arange(len(total_time_by_method))
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.bar(
    x,
    total_time_by_method['partition_time_sum'],
    label='Partition generation',
    color='dodgerblue',
    alpha=0.85,
)
ax.bar(
    x,
    total_time_by_method['explanation_time_sum'],
    bottom=total_time_by_method['partition_time_sum'],
    label='Explanation',
    color='coral',
    alpha=0.85,
)
ax.set_xticks(x)
ax.set_xticklabels(total_time_by_method['method'].astype(str), rotation=25)
ax.set_ylabel('Total seconds across images')
ax.set_title('Total time by method: partition generation + explanation')
ax.grid(axis='y', alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(total_time_plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved total time table: {total_time_table_path}')
print(f'Saved total time plot: {total_time_plot_path}')

## Create XAI Results Webpage
Use saved XAI figures, AUC curves, and prediction JSON files.

In [ ]:
## Create HTML webpage from saved XAI/AUC images and prediction JSON files
from pathlib import Path
import html
import json
import pandas as pd

xai_results_root = Path(path_results) if 'path_results' in globals() else Path(
    '/Users/rashid/data/PhD/datacloud_data/repos/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/results/xai_results'
)
xai_results_root = xai_results_root.expanduser().resolve()
html_report_path = xai_results_root / 'xai_results_report.html'

if not xai_results_root.exists():
    raise FileNotFoundError(f'XAI results folder not found: {xai_results_root}')

def escape(value):
    return html.escape(str(value))

def image_id_12(value):
    text = str(value)
    if text.endswith('.0'):
        text = text[:-2]
    return text.zfill(12)

def fmt_float(value, digits=4):
    if value is None:
        return ''
    try:
        return f'{float(value):.{digits}f}'
    except (TypeError, ValueError):
        return escape(value)

def rel_path(path):
    return escape(path.relative_to(xai_results_root).as_posix())

def image_tag(path, label):
    if path is not None and path.exists():
        return f'<img src="{rel_path(path)}" alt="{escape(label)}" loading="lazy">'
    missing = path.name if path is not None else 'not found'
    return f'<div class="missing">Missing image<br>{escape(missing)}</div>'

def prediction_summary(pred):
    top_classes = pred.get('top_k_classes', []) or []
    top_class = top_classes[0] if top_classes else {}
    return {
        'image_id': image_id_12(pred.get('image_id', '')),
        'fixed_category': pred.get('fixed_category', ''),
        'explained_class': pred.get('explained_class', ''),
        'top_class': top_class.get('class_name', ''),
        'top_confidence': top_class.get('confidence', None),
        'f_S': pred.get('f_S', None),
        'f_0': pred.get('f_0', None),
        'has_segmentation': pred.get('has_segmentation', ''),
    }

def top_classes_table(top_classes):
    if not top_classes:
        return '<p class="muted">No top-k classes saved.</p>'
    rows = []
    for rank, item in enumerate(top_classes, start=1):
        rows.append(
            '<tr>'
            f'<td>{rank}</td>'
            f'<td>{escape(item.get("class_id", ""))}</td>'
            f'<td>{escape(item.get("class_name", ""))}</td>'
            f'<td>{fmt_float(item.get("confidence"), 4)}</td>'
            '</tr>'
        )
    return '<table><thead><tr><th>Rank</th><th>Class ID</th><th>Class</th><th>Confidence</th></tr></thead><tbody>' + ''.join(rows) + '</tbody></table>'

def speed_table(speed):
    if not isinstance(speed, dict) or not speed:
        return '<p class="muted">No speed values saved.</p>'
    rows = ''.join(f'<tr><td>{escape(k)}</td><td>{fmt_float(v, 3)}</td></tr>' for k, v in speed.items())
    return '<table><thead><tr><th>Stage</th><th>ms</th></tr></thead><tbody>' + rows + '</tbody></table>'

def fmt_table_value(value, column=None):
    if value is None or pd.isna(value):
        return ''
    if column in {'auc_ins', 'auc_del', 'f_S', 'f_0'}:
        return fmt_float(value, 4)
    if column in {'time_exp', 'time_eval'}:
        return fmt_float(value, 2)
    if column in {'image_no', 'image_id'}:
        try:
            return str(int(value))
        except (TypeError, ValueError):
            return escape(value)
    return escape(value)

def dataframe_to_html_table(df, columns=None):
    if df is None or df.empty:
        return '<p class="muted">No rows available.</p>'
    view = df.copy()
    if columns is not None:
        view = view[[col for col in columns if col in view.columns]]
    header = ''.join(f'<th>{escape(col)}</th>' for col in view.columns)
    rows = []
    for _, row in view.iterrows():
        rows.append('<tr>' + ''.join(f'<td>{fmt_table_value(row[col], col)}</td>' for col in view.columns) + '</tr>')
    return f'<table><thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody></table>'

def latest_matching_file(folder, pattern, preferred_name=None):
    preferred = folder / preferred_name if preferred_name else None
    if preferred is not None and preferred.exists():
        return preferred
    matches = sorted(folder.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0] if matches else None

items = []
for pred_path in sorted(xai_results_root.glob('*/*_predictions.json')):
    image_dir = pred_path.parent
    with pred_path.open('r') as f:
        pred = json.load(f)

    folder_id = image_dir.name
    img12 = image_id_12(pred.get('image_id', folder_id))
    xai_path = next(iter(sorted(image_dir.glob('*_xai_results.png'))), image_dir / f'{img12}_xai_results.png')
    auc_path = next(iter(sorted(image_dir.glob('auc_results_*.png'))), image_dir / f'auc_results_{folder_id}.png')
    summary = prediction_summary(pred)
    summary['folder_id'] = folder_id
    items.append({
        'folder_id': folder_id,
        'image_id': img12,
        'pred': pred,
        'summary': summary,
        'xai_path': xai_path,
        'auc_path': auc_path,
    })

if not items:
    raise FileNotFoundError(f'No *_predictions.json files found below: {xai_results_root}')

auc_all_path = latest_matching_file(xai_results_root, 'auc_results_all_images_*.csv', 'auc_results_all_images_500_51.csv')
auc_summary_path = latest_matching_file(xai_results_root, 'auc_results_summary_*.csv', 'auc_results_summary_500_51.csv')
auc_boxplot_path = latest_matching_file(xai_results_root, 'auc_results_boxplots_*.png', 'auc_results_boxplots_500_51.png')

auc_all_df_report = pd.read_csv(auc_all_path) if auc_all_path is not None else pd.DataFrame()
auc_summary_df_report = pd.read_csv(auc_summary_path) if auc_summary_path is not None else pd.DataFrame()
if not auc_all_df_report.empty:
    auc_all_df_report['image_id_str'] = auc_all_df_report['image_id'].apply(image_id_12)

auc_columns = ['image_no', 'image_id', 'fixed_category', 'method', 'auc_ins', 'auc_del', 'time_exp', 'time_eval']
auc_top_html = f'''
<section class="top-card">
  <h2>Aggregate AUC Results</h2>
  <p class="muted">AUC table: {escape(auc_all_path.name if auc_all_path is not None else 'not found')}</p>
  <p class="muted">AUC summary: {escape(auc_summary_path.name if auc_summary_path is not None else 'not found')}</p>
  <figure>
    <figcaption>Aggregate AUC box plot</figcaption>
    {image_tag(auc_boxplot_path, 'Aggregate AUC box plot')}
  </figure>
  <h3>Aggregate Summary</h3>
  <div class="table-scroll small-scroll">{dataframe_to_html_table(auc_summary_df_report)}</div>
  <details class="auc-details">
    <summary>Full AUC Table</summary>
    <div class="table-scroll full-auc-table">{dataframe_to_html_table(auc_all_df_report, auc_columns)}</div>
  </details>
</section>'''

def auc_results_table(image_id):
    if auc_all_df_report.empty:
        return '<p class="muted">No aggregate AUC table found.</p>'
    sub = auc_all_df_report.loc[auc_all_df_report['image_id_str'] == image_id].copy()
    if sub.empty:
        return '<p class="muted">No AUC rows found for this image.</p>'
    method_order = ['BPT', 'sam', 'coverage', 'compact', 'filled', 'refined']
    sub['method'] = pd.Categorical(sub['method'], categories=method_order, ordered=True)
    sub = sub.sort_values('method')
    return dataframe_to_html_table(sub, ['method', 'auc_ins', 'auc_del', 'time_exp', 'time_eval'])

overview_rows = []
sections = []
for item in items:
    s = item['summary']
    overview_rows.append(
        f'<tr data-image="{escape(item["image_id"])}" data-folder="{escape(item["folder_id"])}">'
        f'<td><a href="#{escape(item["image_id"])}">{escape(item["image_id"])}</a></td>'
        f'<td>{escape(s["fixed_category"])}</td>'
        f'<td>{escape(s["explained_class"])}</td>'
        f'<td>{escape(s["top_class"])}</td>'
        f'<td>{fmt_float(s["top_confidence"], 4)}</td>'
        f'<td>{fmt_float(s["f_S"], 4)}</td>'
        f'<td>{fmt_float(s["f_0"], 4)}</td>'
        f'<td>{escape(s["has_segmentation"])}</td>'
        '</tr>'
    )

    pred = item['pred']
    detail_rows = ''.join(
        f'<tr><td>{escape(key)}</td><td>{escape(value)}</td></tr>'
        for key, value in s.items()
        if key != 'folder_id'
    )
    sections.append(f'''
<section class="result-card" id="{escape(item['image_id'])}" data-image="{escape(item['image_id'])}" data-folder="{escape(item['folder_id'])}">
  <div class="section-title">
    <h2>Image {escape(item['image_id'])}</h2>
    <a href="#{escape(item['image_id'])}">#{escape(item['folder_id'])}</a>
  </div>
  <details class="prediction-details">
    <summary>Prediction details</summary>
    <div class="details-grid">
      <div>
        <h3>Prediction Details</h3>
        <table><tbody>{detail_rows}</tbody></table>
      </div>
      <div>
        <h3>Top-k Classes</h3>
        {top_classes_table(pred.get('top_k_classes', []))}
      </div>
      <div>
        <h3>YOLO Speed</h3>
        {speed_table(pred.get('speed', {}))}
      </div>
    </div>
  </details>
  <details class="auc-details auc-image-table">
    <summary>AUC results for this image</summary>
    {auc_results_table(item['image_id'])}
  </details>
  <figure>
    <figcaption>XAI results</figcaption>
    {image_tag(item['xai_path'], item['image_id'] + ' XAI results')}
  </figure>
  <figure>
    <figcaption>AUC curves</figcaption>
    {image_tag(item['auc_path'], item['image_id'] + ' AUC curves')}
  </figure>
</section>''')

html_doc = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>XAI Results Report</title>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; margin: 0; color: #1f2933; background: #f7f8fa; }}
  header {{ padding: 24px 32px; background: #111827; color: white; }}
  h1, h2, h3 {{ margin: 0; }}
  header p {{ margin: 8px 0 0; color: #cbd5e1; }}
  main {{ padding: 24px 32px 48px; }}
  .controls {{ margin: 18px 0; }}
  input {{ width: min(460px, 100%); padding: 10px 12px; border: 1px solid #cbd5e1; border-radius: 8px; font-size: 14px; }}
  .overview-wrap {{ max-height: 420px; overflow: auto; border: 1px solid #e5e7eb; border-radius: 8px; background: white; }}
  .table-scroll {{ overflow: auto; border: 1px solid #e5e7eb; border-radius: 8px; background: white; margin: 10px 0 18px; }}
  .small-scroll {{ max-height: 260px; }}
  .full-auc-table {{ max-height: 520px; }}
  table {{ border-collapse: collapse; width: 100%; background: white; }}
  th, td {{ padding: 8px 10px; border-bottom: 1px solid #e5e7eb; text-align: left; font-size: 13px; vertical-align: top; }}
  th {{ background: #f1f5f9; position: sticky; top: 0; z-index: 1; }}
  a {{ color: #2563eb; text-decoration: none; }}
  .top-card, .result-card {{ margin-top: 24px; padding: 18px; background: white; border: 1px solid #e5e7eb; border-radius: 8px; }}
  .section-title {{ display: flex; align-items: baseline; justify-content: space-between; gap: 16px; margin-bottom: 14px; }}
  .prediction-details {{ margin-bottom: 16px; border: 1px solid #e5e7eb; border-radius: 8px; background: #fbfdff; }}
  .prediction-details summary {{ cursor: pointer; padding: 10px 12px; font-weight: 700; color: #334155; }}
  .details-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 16px; padding: 0 12px 12px; }}
  .details-grid h3 {{ margin-bottom: 8px; font-size: 15px; }}
  .auc-details {{ margin-bottom: 16px; border: 1px solid #e5e7eb; border-radius: 8px; background: #fbfdff; }}
  .auc-details summary {{ cursor: pointer; padding: 10px 12px; font-weight: 700; color: #334155; }}
  .auc-details .table-scroll, .auc-image-table table {{ margin: 0 12px 12px; width: calc(100% - 24px); }}
  .top-card h3 {{ margin: 12px 0 8px; font-size: 15px; }}
  figure {{ margin: 0 0 16px; border: 1px solid #e5e7eb; border-radius: 8px; overflow: hidden; background: #fff; }}
  figcaption {{ padding: 8px 10px; font-weight: 700; background: #f8fafc; border-bottom: 1px solid #e5e7eb; }}
  img {{ display: block; width: 70%; height: auto; }}
  .missing {{ padding: 32px; text-align: center; color: #b91c1c; background: #fff1f2; }}
  .muted {{ color: #64748b; }}
</style>
</head>
<body>
<header>
  <h1>XAI Results Report</h1>
  <p>{len(items)} images from {escape(xai_results_root)}</p>
</header>
<main>
  <div class="controls">
    <input id="imageFilter" placeholder="Filter by image id, folder id, or class..." oninput="filterImages()">
  </div>
  {auc_top_html}
  <h2>All Images</h2>
  <div class="overview-wrap">
    <table>
      <thead><tr><th>Image</th><th>Fixed category</th><th>Explained class</th><th>Top class</th><th>Top confidence</th><th>f_S</th><th>f_0</th><th>Segmentation</th></tr></thead>
      <tbody>{''.join(overview_rows)}</tbody>
    </table>
  </div>
  {''.join(sections)}
</main>
<script>
function filterImages() {{
  const q = document.getElementById('imageFilter').value.trim().toLowerCase();
  document.querySelectorAll('[data-image]').forEach(el => {{
    const text = el.innerText.toLowerCase() + ' ' + el.dataset.image + ' ' + el.dataset.folder;
    el.style.display = text.includes(q) ? '' : 'none';
  }});
}}
</script>
</body>
</html>
'''

html_report_path.write_text(html_doc, encoding='utf-8')
print(f'HTML report written to: {html_report_path}')
html_report_path